# Активное обучение: разметка только самых информативных объектов

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Активное обучение».

## Что делает метод

Активное обучение — это стратегия, при которой модель сама выбирает наиболее информативные объекты для разметки. Цикл такой: модель обучается на небольшой размеченной выборке, оценивает большой пул неразмеченных данных и по выбранному критерию отбирает объекты для разметки экспертом, после чего цикл повторяется.

Метод применим, когда разметка дорога (требует эксперта, эксперимента, времени), а неразмеченных данных много. В этом блокноте мы сравним активное обучение со случайной разметкой и покажем, что оно достигает того же качества меньшим числом меток. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вы биолог: у вас 5000 изображений клеток, и нужно научить модель их классифицировать. Разметить все изображения (указать, какие клетки здоровые, какие поражены) — дорогостоящий труд патолога. Как выбрать, какие именно изображения разметить в первую очередь?

Неопытный подход: брать случайные изображения. Активное обучение делает умнее:

1. **Обучите модель** на небольшой уже размеченной выборке (например, 20 изображений).
2. **Пусть модель оценит** все оставшиеся неразмеченные изображения: «насколько я уверена в каждом?».
3. **Выберите те изображения, в которых модель наименее уверена** — именно их метка несёт больше всего новой информации.
4. **Разметьте их** (отдайте патологу) и обновите модель. Повторите.

Ключевые понятия:
- **Пул** (pool) — большой набор неразмеченных данных, из которого выбираем объекты для разметки.
- **Выборка/запрос разметки** — объекты, которые модель предлагает разметить на текущем шаге.
- **Неопределённость** (uncertainty) — мера того, насколько модель не уверена в своём прогнозе для объекта.
- **Энтропия** — способ измерить неопределённость: если модель равномерно «делит» вероятность между 10 классами — это максимальная неопределённость. Если 99 % вероятности у одного класса — почти никакой.
- **Кривая обучения** (learning curve) — как меняется качество модели с ростом числа размеченных объектов. Активное обучение даёт более крутую кривую, чем случайный выбор.

## 1. Установка библиотек

В среде Google Colab перечисленные библиотеки, как правило, уже установлены. Ячейка ниже гарантирует их наличие, в том числе при локальном запуске.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",  # фон полотна
    "node_fill":  "#eef1f6",  # заливка блоков
    "node_text":  "#0e1726",  # основной текст
    "edge":       "#4d5e78",  # линии, оси, подписи
    "grid":       "#dce2ec",  # сетка координат
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],  # цвета рядов
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем открытый набор рукописных цифр (`digits` из `scikit-learn`): 1797 изображений 8 на 8 пикселей, десять классов. Большую часть выборки объявим «неразмеченным пулом», малую — стартовой размеченной выборкой, а отдельную часть отложим для честной оценки качества.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True)

# отложенная тестовая выборка для оценки качества
X_pool, X_test, y_pool, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

rng = np.random.default_rng(42)
start = rng.choice(len(X_pool), size=20, replace=False)
print(f'Неразмеченный пул: {len(X_pool)} объектов')
print(f'Стартовая разметка: {len(start)} объектов')
print(f'Тестовая выборка: {len(X_test)} объектов')

## 4. Применение метода

### Стратегия отбора по неопределённости

Критерий отбора — **неопределённость предсказания**. Для каждого объекта пула вычислим **энтропию** распределения вероятностей классов:

```
Энтропия = -Σ (вероятность класса k) × log(вероятность класса k)
```

Чем выше энтропия, тем менее уверена модель в этом объекте. Модель не может отдать предпочтение ни одному классу — значит, метка будет максимально информативна.

**Логистическая регрессия** (`LogisticRegression`) — базовый, но эффективный классификатор. Она возвращает калиброванные вероятности классов, что делает энтропию осмысленной. `StandardScaler` масштабирует признаки: для логистической регрессии это важно, чтобы признаки с разными масштабами не доминировали.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

def new_model():
    return make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=2000))

def entropy(proba):
    """Энтропия распределения вероятностей классов."""
    p = np.clip(proba, 1e-12, 1.0)
    return -np.sum(p * np.log(p), axis=1)

def run_active_learning(strategy, n_rounds=18, batch=15):
    """Цикл активного обучения; strategy = 'uncertainty' или
    'random'. Возвращает число меток и качество на тесте."""
    labeled = list(start)
    pool = [i for i in range(len(X_pool)) if i not in set(start)]
    sizes, scores = [], []
    local_rng = np.random.default_rng(0)
    for _ in range(n_rounds):
        model = new_model()
        model.fit(X_pool[labeled], y_pool[labeled])
        sizes.append(len(labeled))
        scores.append(accuracy_score(y_test,
                                     model.predict(X_test)))
        if not pool:
            break
        if strategy == 'uncertainty':
            proba = model.predict_proba(X_pool[pool])
            order = np.argsort(entropy(proba))[::-1]
            chosen = [pool[i] for i in order[:batch]]
        else:
            chosen = list(local_rng.choice(pool,
                          size=min(batch, len(pool)),
                          replace=False))
        labeled += chosen
        pool = [i for i in pool if i not in set(chosen)]
    return sizes, scores

print('Функция цикла активного обучения готова.')

### Сравнение со случайной разметкой

Запустим два цикла: с отбором по неопределённости (активное обучение) и со случайным отбором того же числа объектов. Параметры одинаковы: стартовая выборка из 20 объектов, 18 раундов по 15 новых меток.

Сравнение честное: оба метода видят одинаковое число меток на каждом шаге — разница только в том, *какие* объекты выбираются.

In [ ]:
sizes_al, scores_al = run_active_learning('uncertainty')
sizes_rnd, scores_rnd = run_active_learning('random')

print('Активное обучение и случайная разметка завершены.')
print(f'Меток в конце: {sizes_al[-1]}')
print(f'Качество, активное обучение: {scores_al[-1]:.3f}')
print(f'Качество, случайная разметка: {scores_rnd[-1]:.3f}')

### Кривые обучения: активное обучение против случайного

Построим зависимость точности классификации на независимой тестовой выборке от числа размеченных объектов. Если активное обучение работает, его кривая должна идти выше — то же качество достигается меньшим числом меток.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9.8, 5.6))
ax.plot(sizes_al, scores_al, marker='o', color=VIZ['series'][0],
        label='активное обучение (по неопределённости)')
ax.plot(sizes_rnd, scores_rnd, marker='s', color=VIZ['series'][2],
        label='случайная разметка')
ax.set_xlabel('Число размеченных объектов')
ax.set_ylabel('Доля верных ответов на тесте')
ax.set_title('Активное обучение против случайной разметки')
ax.legend()
fig.tight_layout()
plt.show()

**Как читать этот график.**

Горизонтальная ось — число размеченных объектов, потраченных на обучение. Вертикальная ось — доля верных ответов на независимой тестовой выборке (1.0 = 100 %).

На что обратить внимание:
1. Если синяя кривая (активное обучение) идёт выше оранжевой (случайный выбор) — метод экономит разметку.
2. **Горизонтальный зазор** между кривыми на одном уровне точности — это экономия: сколько меток нам не нужно ставить.
3. К концу кривые могут сойтись: когда данных достаточно, стратегия выбора не важна. Выгода активного обучения — в начале пути, при малом бюджете разметки.

### Какие объекты отбирает метод

Покажем конкретные изображения цифр с наибольшей неопределённостью на старте обучения — именно их активное обучение предлагает разметить в первую очередь. Как правило, это нетипично или неразборчиво написанные цифры, находящиеся «на границе» между несколькими классами.

In [ ]:
model = new_model().fit(X_pool[start], y_pool[start])
rest = np.array([i for i in range(len(X_pool))
                 if i not in set(start)])
unc = entropy(model.predict_proba(X_pool[rest]))
hard = rest[np.argsort(unc)[::-1][:8]]

fig, axes = plt.subplots(2, 4, figsize=(11.0, 6.0))
for ax, idx in zip(axes.ravel(), hard):
    ax.imshow(X_pool[idx].reshape(8, 8), cmap='Blues')
    pred = model.predict(X_pool[idx:idx + 1])[0]
    ax.set_title(f'прогноз: {pred}, факт: {y_pool[idx]}',
                 fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
fig.suptitle('Объекты с наибольшей неопределённостью', fontsize=14)
fig.tight_layout()
plt.show()

**Как читать этот график.**

Каждое маленькое изображение — цифра, которую модель при текущем обучении не смогла уверенно классифицировать. Над каждым изображением: прогноз модели и истинный класс. Если прогноз не совпадает с фактом — это «трудный» объект. Именно такие объекты несут максимальную информацию: их разметка исправит наиболее критичные ошибки модели.

## 5. Интерпретация результата

- **Кривые обучения**: кривая активного обучения, как правило, идёт выше кривой случайной разметки — то же качество достигается меньшим числом меток. Горизонтальный зазор — это экономия ручной работы.
- **Отобранные объекты**: метод запрашивает метки для пограничных и нетипичных объектов — тех, что сильнее всего уточнят границы между классами.
- **Энтропия** как критерий: объекты с равномерным распределением вероятностей по классам (модель «пожимает плечами») — самые информативные. Объекты с высокой уверенностью (99 % у одного класса) — малоинформативные.
- **Отложенный тестовый набор** строго необходим: активный отбор смещает обучающую выборку, и оценивать качество на ней нельзя.
- Выигрыш активного обучения не гарантирован при сильном шуме разметки или очень маленькой начальной выборке — тогда первые запросы почти случайны.

## 6. Попробуйте на своих данных

Замените демонстрационный набор своими данными. Представьте, что размечена лишь малая часть, а остальное — пул для разметки.

1. Загрузите файл в Colab через вкладку «Файлы».
2. Снимите комментарии в ячейке ниже и укажите имя файла и столбец с меткой.
3. Выполните ячейки раздела 4.

## 7. Поэкспериментируйте сами

Попробуйте изменить следующие параметры функции `run_active_learning` (раздел 4):

- **`n_rounds`**: уменьшите до 5 или увеличьте до 30. Как меняется финальная точность?
- **`batch`**: попробуйте 1 (разметка по одному объекту), 50 (крупные пакеты). Обратите внимание: большие пакеты могут выбирать похожие объекты (избыточность); маленькие — дольше, зато точнее.
- **Стартовая выборка (`size=20`)**: попробуйте `size=5` или `size=100`. При очень маленьком старте активное обучение работает хуже — почему?
- **Другой критерий**: замените в стратегии `entropy` на критерий «наименьшая вероятность наиболее вероятного класса»: `1 - proba.max(axis=1)`. Меняется ли результат?

In [ ]:
# --- Шаблон загрузки своих данных ---
# df = pd.read_csv('my_data.csv')
# target_column = 'класс'
#
# y_all = df[target_column].astype('category').cat.codes.to_numpy()
# X_all = df.drop(columns=[target_column])\
#           .select_dtypes('number').to_numpy()
#
# from sklearn.model_selection import train_test_split
# X_pool, X_test, y_pool, y_test = train_test_split(
#     X_all, y_all, test_size=0.3, random_state=42, stratify=y_all)
# start = rng.choice(len(X_pool), size=20, replace=False)
# Далее повторите ячейки раздела 4.

## Краткие выводы

- Активное обучение — стратегия умной разметки: модель сама указывает, какие объекты разметить в первую очередь, чтобы получить максимальный прирост качества.
- Отбор ведётся по **неопределённости** (энтропии): объекты, в которых модель наименее уверена, несут наибольшую информацию.
- На кривых обучения активное обучение обычно достигает того же качества при **меньшем числе меток**, чем случайный выбор. Горизонтальный зазор — это сэкономленный труд эксперта.
- **Трудные объекты** (неразборчивые, нетипичные) — это именно те, что активное обучение выбирает первыми. Их метки максимально уточняют границы между классами.
- Качество нужно оценивать строго на **отдельном тестовом наборе**: обучающая выборка смещена, и оценка на ней завышает качество.
- Метод не гарантирует выигрыша при маленьком старте или зашумлённой разметке — начинайте с разумной стартовой выборки (не менее 2–3 объектов на класс).

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На кривых обучения синяя линия (активное обучение) в начале идёт заметно выше оранжевой, но к концу они сближаются. Что это означает для вашего реального проекта, если бюджет разметки ограничен, скажем, 50–80 объектами?
2. Стартовая выборка составляет всего 20 объектов из 10 классов цифр. Чем рискует исследователь, если в этих 20 объектах некоторые классы вообще не представлены?
3. Вы перешли с энтропии на критерий «наименьшая вероятность лидирующего класса» (1 − max p_k) и результат почти не изменился. Означает ли это, что оба критерия эквивалентны, или есть ситуации, где они могут разойтись?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Именно в этом диапазоне (малый бюджет) выигрыш активного обучения максимален: то же качество достигается при существенно меньшем числе меток. Если бюджет неограничен, стратегия выбора теряет значение — при достаточном количестве данных случайная выборка догоняет активную.
2. Если класс не попал в стартовую выборку, модель не знает о его существовании и не может отметить его объекты как неопределённые. Первые запросы активного обучения пропустят этот класс целиком — ошибка распространится на несколько раундов. Это называется проблемой «холодного старта»: минимально необходимо хотя бы по 2–3 примера каждого класса.
3. Для двух классов оба критерия математически совпадают. При большом числе классов они могут расходиться: энтропия учитывает всё распределение (в том числе несколько равновероятных классов), тогда как margin учитывает только разрыв между первым и вторым. В сложных ситуациях с множеством конкурирующих классов энтропия информативнее.
</details>
